# Фазовые траектории лучших особей в ходе обучения

Полная модель **Afferents**, шаг **0.1 мс**, физический мышечный момент **без ×10**.
Во всех воспроизведениях `mode=0`: конечностью управляет нейромышечная сеть.

Тетрадка читает **популяции каждого сохранённого поколения**, ранжирует их по
training fitness и заново интегрирует динамику. Лучший по training в одном
поколении может отличаться от победителя итоговой validation. Сравниваемые
геномы разных поколений не обязательно принадлежат одной линии наследования.
Точки фазовой плоскости — проекции полной системы: нейронные, мышечные и
афферентные состояния при каждом запуске сбрасываются одинаковым образом.

Выберите ядро **SpikingNN (.venv)** и выполните ячейки сверху вниз.
Запуск возможен из корня репозитория или любого вложенного каталога.

In [ ]:
from pathlib import Path
import json, sys

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/SpikingNN/core/Izh_net.py").exists()), None)
if REPO is None:
    raise RuntimeError("Запустите Jupyter внутри репозитория SpikingNN")
sys.path.insert(0, str(REPO))
from scripts.AfferentBalance2026.runtime_support import (
    BASE, current_results, runtime_directory, activate_runtime, core_module)
# Для другой серии задайте ROOT явно вместо чтения указателя.
ROOT = current_results()
manifest = json.loads((ROOT / "manifest.json").read_text())
RUNTIME = activate_runtime(runtime_directory(ROOT))
sys.path.insert(0, str(BASE / "notebooks"))
MECHANICS = core_module("config").MECHANICS
DT_MS = core_module("config").DT_MS
assert MECHANICS.muscle_torque_factor == 1. and DT_MS == .1
print("Серия:", ROOT, "\nСтатус:", manifest["status"], "\nЯдро:", RUNTIME)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from phase_viewer import (configure_fonts, catalog, initial_cases, draw,
                          compare_generations, learning_plot, metric_table, widget_viewer, validation_snapshot)
load_generation = core_module("training_history").load_generation
print("Шрифт:", configure_fonts())
RUNS = catalog(ROOT)
if not RUNS:
    raise RuntimeError("Ещё нет сохранённых поколений: дождитесь начала обучения")
for name, entry in RUNS.items():
    print(f"{name}: {len(entry['generations'])} поколений, последнее {max(entry['generations'])}")

## Пример: как изменяются решения в ходе обучения

По умолчанию выбран A1, seed=43; если он ещё не рассчитан, берётся первая доступная
серия. Измените `RUN_NAME` для другого опыта. Отбор здесь только по training,
тестовые начальные условия не меняют геномы и не участвуют в обучении.
В таблице `late_rmse_deg=90` у упавшей особи означает штраф метрики, а не
измеренную ошибку установившегося движения.

In [ ]:
RUN_NAME = "A1/seed_42" if "A1/seed_42" in RUNS else next(iter(RUNS))
ENTRY = RUNS[RUN_NAME]
FILES = ENTRY["generations"]
print("Выбрано:", RUN_NAME)
fig = learning_plot(ENTRY["path"])
display(fig); plt.close(fig)
GENERATIONS = sorted(set([min(FILES), min(FILES, key=lambda n: abs(n-60)), max(FILES)]))
CASES = initial_cases("mixed", target_deg=0.)
fig = compare_generations(FILES, GENERATIONS, CASES)
display(fig); plt.close(fig)

## Последнее поколение: полные траектории и экспорт

Окружность — начальное состояние; квадрат — конец успешного испытания;
крест — конец неуспешного. Чёрная звезда — целевая точка.
Вычисления сохраняют каждый шаг; досрочное падение не дорисовывается до конца.
Артефакты записываются в игнорируемый Git каталог `report/`.

Можно задать собственный массив `CASES`: каждая строка содержит
`[целевой угол (рад), начальный угол (рад), скорость (рад/с), время толчка (с),
скачок скорости (рад/с)]`; время −1 означает отсутствие толчка.

In [ ]:
EXPORT = ROOT / "notebook_exports" / RUN_NAME.replace("/", "__")
fig, results = draw(FILES[max(FILES)], CASES, ranks=(1,), export=EXPORT, time_window_s=1.)
display(fig); plt.close(fig)
try:
    import pandas as pd
    display(pd.DataFrame(metric_table(results)))
except ImportError:
    print(json.dumps(metric_table(results), ensure_ascii=False, indent=2))
print("Экспорт:", EXPORT)

## Победитель по validation

Ниже показана особь, выбранная на последней завершённой контрольной эпохе.
Её training-ранг может отличаться от 1. Выбор берётся из `selected_index`
сохранённой конечной популяции, повторного отбора по новым траекториям нет.

In [ ]:
validation_choice = validation_snapshot(ENTRY)
if validation_choice is not None:
    validation_file, validation_training_rank = validation_choice
    print("Победитель validation; training-ранг:", validation_training_rank)
    fig, validation_results = draw(validation_file, CASES,
                                  ranks=(validation_training_rank,), time_window_s=1., selection_label="validation")
    display(fig); plt.close(fig)
else:
    print("Validation ещё не выполнена: доступны только training-снимки.")

## Интерактивное исследование

Выберите обучение, поколение, количество лучших особей и набор стартовых точек,
затем нажмите **«Рассчитать траектории»**. «Окно, с» меняет только масштаб графика времени; симуляция всегда полная. Для равновесия целевой угол фиксирован
на нуле. Для обученных законов тока можно менять командный угол: соответствующие
токи вычисляются из сохранённых коэффициентов, оставаясь постоянными во времени.

После появления новых результатов выполните эту ячейку повторно для обновления списка.

In [ ]:
viewer = widget_viewer(ROOT)

## Пять целевых углов и далёкие начальные положения

Это дополнительная проверка выбранного по validation закона тока;
до завершения validation используется training-лидер последнего поколения; она не выполняет повторный отбор по этим пяти испытаниям.
Начальные ошибки достигают 90°. Если обучение токов ещё идёт, используется
последнее доступное поколение выбранной степени.

In [ ]:
command_runs = [name for name in RUNS if "/polynomial/degree_" in name]
if command_runs:
    selection_file = ROOT / "angle_setpoint/polynomial/selection.json"
    if selection_file.exists():
        chosen = json.loads(selection_file.read_text())["overall"]
        LAW_RUN = "angle_setpoint/polynomial/" + chosen
    else:
        LAW_RUN = command_runs[0]
    law_files = RUNS[LAW_RUN]["generations"]
    TARGETS = np.array([-45., -22.5, 0., 22.5, 45.])
    STARTS = np.array([-40., -0.5, 0., 22.5, 45.])
    LAW_CASES = np.column_stack((np.deg2rad(TARGETS), np.deg2rad(STARTS),
                                [.15,-.15,.15,-.15,.15], np.full(5,-1.), np.zeros(5)))
    choice = validation_snapshot(RUNS[LAW_RUN])
    law_file, law_rank = choice if choice is not None else (law_files[max(law_files)], 1)
    print(LAW_RUN, "validation" if choice is not None else "training", "training-ранг", law_rank)
    fig, law_results = draw(law_file, LAW_CASES, ranks=(law_rank,),
                           selection_label="validation" if choice is not None else "training",
                           export=ROOT / "notebook_exports/five_targets")
    display(fig); plt.close(fig)
    #print(json.dumps(metric_table(law_results), ensure_ascii=False, indent=2))
else:
    print("Обучение законов задающих токов ещё не сохранило поколения.")

## Происхождение данных и воспроизводимость

- `manifest.json` — статус и состав полного пересчёта; `factor_1/runtime/` —
  неизменяемая копия исходников. Тетрадка использует именно её.
- `generations/generation_XXXX.npz` — вся популяция, физические параметры,
  коэффициенты токов, оценки и метрики training, параметры механики и хеши ядра.
- `epochs_*/` — контрольные срезы **одного продолжающегося** обучения до 200 эпох;
  тетрадка объединяет их поколения без повторения и без интерполяции геномов.
- `final_population.npz` — последняя популяция и победитель по validation;
  `history.csv` — агрегированная история. Это отдельный от training-рейтинг.
- `notebook_exports/` — изображения, CSV метрик, полные NPZ траекторий и
  `provenance.json` с хешем исходного поколения. Эти файлы не включаются в Git.

Полная схема и команды: [README](README.md). Устойчивость показана только
на выполненных испытаниях; проекция траектории на (θ,ω) не доказывает
глобальную устойчивость многомерной нейромышечной системы.